<a href="https://colab.research.google.com/github/maximofernandezriera/000-thinkpython/blob/main/clase-cap-10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# La base en Marte: diccionarios para encontrarlo todo por su nombre

Quinta parte del repaso de *Think Python*, ahora con el capítulo 10. Ya tenemos la tripulación y las muestras en listas, pero ahora toca montar la base: un inventario de oxígeno, agua y comida, el recuento de rocas de cada tipo y quién trabaja en cada módulo. Con una lista tendríamos que recordar que el agua está en la posición 1; con un **diccionario** basta con pedir `"agua"`.

Mismo formato de siempre: yo escribo, vosotros predecís, y las celdas que fallan lo hacen a propósito.

## Parte 10 — Diccionarios (capítulo 10)

### Un diccionario es un mapeo

En una lista los índices son siempre enteros: 0, 1, 2... En un diccionario los índices, que se llaman **claves**, pueden ser de (casi) cualquier tipo. Recordad la lista de la clase anterior:

In [ ]:
tripulacion = ["Ada", "Mae", "Valentina", "Yuri"]
tripulacion[1]

Con una lista vamos de la posición al valor. Pero en la base queremos ir del **nombre** al valor: ¿cuántos kilos de agua nos quedan? Empezamos con un diccionario vacío, que se escribe con llaves:

In [ ]:
inventario = {}
inventario

Para añadir un **elemento** usamos corchetes, como con las listas, pero dentro va la clave. Cada elemento es una pareja **clave: valor**.

In [ ]:
inventario["oxigeno"] = 120     # kg
inventario

In [ ]:
inventario["agua"] = 340
inventario["comida"] = 85
inventario

Para consultar un valor, se pide por su clave. Antes de ejecutar: ¿qué sale?

In [ ]:
inventario["agua"]

¿Y si pedimos algo que no hemos cargado en la nave?

In [ ]:
inventario["cafe"]

`KeyError: 'cafe'`. Es el equivalente al `IndexError` de las listas: la clave no existe. Y `len` cuenta los elementos, es decir, las parejas clave-valor:

In [ ]:
len(inventario)

En matemáticas esto es un **mapeo**: cada clave *mapea a* un valor. El esquema de la pizarra compara las dos formas de guardar las cosas:

![Pizarra: lista frente a diccionario](https://raw.githubusercontent.com/maximofernandezriera/000-thinkpython/main/pizarra-diccionarios.png)

### Crear diccionarios

No hace falta ir elemento a elemento: se puede escribir el diccionario entero de una vez, con las parejas separadas por comas.

In [ ]:
inventario = {"oxigeno": 120, "agua": 340, "comida": 85}
inventario

La función `dict` crea un diccionario vacío, y si le pasamos otro diccionario, crea una **copia**:

In [ ]:
vacio = dict()
vacio

Esto conecta con el *aliasing* del capítulo 9: los diccionarios son **mutables**, así que `b = a` no copia, crea un alias. Antes de cada simulación de consumo guardamos un respaldo del inventario. ¿Qué vale `respaldo["agua"]` al final? ¿Y `alias["agua"]`?

In [ ]:
respaldo = dict(inventario)    # copia
alias = inventario             # alias: el mismo diccionario
inventario["agua"] -= 40       # un sol de consumo
inventario["agua"], respaldo["agua"], alias["agua"]

[▶ Ver este ejemplo paso a paso en PythonTutor](https://pythontutor.com/render.html#code=inventario%20%3D%20%7B%22oxigeno%22%3A%20120%2C%20%22agua%22%3A%20340%2C%20%22comida%22%3A%2085%7D%0Arespaldo%20%3D%20dict%28inventario%29%0Aalias%20%3D%20inventario%0Ainventario%5B%22agua%22%5D%20-%3D%2040%0Aprint%28respaldo%5B%22agua%22%5D%2C%20alias%5B%22agua%22%5D%29&mode=edit&py=3)

### El operador `in`

`in` también funciona con diccionarios, pero mira **solo las claves**. Predecid las dos respuestas:

In [ ]:
"agua" in inventario, 120 in inventario

`120` es un valor, no una clave, así que da `False`. Para buscar entre los valores existe el método `values`:

In [ ]:
120 in inventario.values()

### Por qué los diccionarios son tan rápidos

Python guarda los diccionarios en una **tabla hash**: buscar una clave tarda más o menos lo mismo tenga el diccionario 3 elementos o 100 000. En una lista, `in` tiene que mirar los elementos uno a uno. Lo comprobamos con la lista de palabras del capítulo 9.

In [ ]:
from os.path import basename, exists

def descargar(url):
    archivo = basename(url)
    if not exists(archivo):
        from urllib.request import urlretrieve
        urlretrieve(url, archivo)
        print("Descargado", archivo)
    return archivo

descargar("https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt")
palabras = open("words.txt").read().split()
len(palabras)

El ordenador de a bordo quiere contar los pares de palabras que son una la inversa de la otra, como *stressed* y *desserts*. Primera versión, buscando en la **lista**:

In [ ]:
def invertir(palabra):
    return "".join(reversed(palabra))

def demasiado_lento():
    cuenta = 0
    for palabra in palabras:
        if invertir(palabra) in palabras:     # in sobre una lista: mira una a una
            cuenta += 1
    return cuenta

# %time demasiado_lento()    # tarda más de un minuto: mejor no lo ejecutéis en clase

El `in` está dentro del bucle, así que para cada una de las más de 100 000 palabras recorremos la lista entera otra vez. ¿Cuántas comparaciones son, más o menos?

In [ ]:
len(palabras) ** 2

Casi 13 000 millones. Ahora metemos las palabras como claves de un diccionario. Los valores no importan (ponemos `1`), solo queremos saber si una clave existe:

In [ ]:
diccionario_palabras = {}
for palabra in palabras:
    diccionario_palabras[palabra] = 1

def mucho_mas_rapido():
    cuenta = 0
    for palabra in diccionario_palabras:
        if invertir(palabra) in diccionario_palabras:   # in sobre un diccionario: directo
            cuenta += 1
    return cuenta

%time mucho_mas_rapido()

Una fracción de segundo, frente a más de un minuto. `%time` es un *comando mágico* de Jupyter/Colab: no es Python, pero sirve para medir cuánto tarda una celda.

### Una colección de contadores

El rover vuelve con una bolsa de muestras y queremos saber cuántas hay de cada tipo. Un diccionario es la herramienta perfecta: una clave por tipo de roca y, como valor, cuántas hemos visto. El patrón es *inicializar, recorrer, actualizar*, igual que en `mas_frio`:

In [ ]:
def contar(secuencia):
    contador = {}
    for x in secuencia:
        if x not in contador:
            contador[x] = 1       # primera vez que lo vemos
        else:
            contador[x] += 1      # ya estaba: sumamos uno
    return contador

muestras = ["basalto", "olivino", "basalto", "hematita", "basalto", "olivino"]
contar(muestras)

[▶ Ver este ejemplo paso a paso en PythonTutor](https://pythontutor.com/render.html#code=def%20contar%28secuencia%29%3A%0A%20%20%20%20contador%20%3D%20%7B%7D%0A%20%20%20%20for%20x%20in%20secuencia%3A%0A%20%20%20%20%20%20%20%20if%20x%20not%20in%20contador%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20contador%5Bx%5D%20%3D%201%0A%20%20%20%20%20%20%20%20else%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20contador%5Bx%5D%20%2B%3D%201%0A%20%20%20%20return%20contador%0A%0Amuestras%20%3D%20%5B%22basalto%22%2C%20%22olivino%22%2C%20%22basalto%22%2C%20%22hematita%22%5D%0Aprint%28contar%28muestras%29%29&mode=edit&py=3)

Como recorre cualquier secuencia, `contar` también funciona con strings. Antes de ejecutar: ¿qué letra aparece más veces en el nombre del rover?

In [ ]:
contar("curiosity")

### Recorrer un diccionario

Un `for` sobre un diccionario recorre sus **claves**:

In [ ]:
for recurso in inventario:
    print(recurso)

Para los valores, el método `values`. Y para las dos cosas a la vez, recorremos las claves y buscamos cada valor (en el próximo capítulo veremos una forma más corta):

In [ ]:
for kg in inventario.values():
    print(kg)

In [ ]:
for recurso in inventario:
    print(recurso, inventario[recurso], "kg")

Juntando lo de hoy con las listas del capítulo 9: ¿qué recursos están por debajo de un límite? Recorremos el diccionario y **acumulamos** una lista con las claves que cumplen la condición. Esto se llama **filtrar**.

In [ ]:
def bajo_minimos(inventario, limite):
    alerta = []
    for recurso in inventario:
        if inventario[recurso] < limite:
            alerta.append(recurso)
    return alerta

bajo_minimos(inventario, 150)

[▶ Ver este ejemplo paso a paso en PythonTutor](https://pythontutor.com/render.html#code=def%20bajo_minimos%28inventario%2C%20limite%29%3A%0A%20%20%20%20alerta%20%3D%20%5B%5D%0A%20%20%20%20for%20recurso%20in%20inventario%3A%0A%20%20%20%20%20%20%20%20if%20inventario%5Brecurso%5D%20%3C%20limite%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20alerta.append%28recurso%29%0A%20%20%20%20return%20alerta%0A%0Ainventario%20%3D%20%7B%22oxigeno%22%3A%20120%2C%20%22agua%22%3A%20300%2C%20%22comida%22%3A%2085%7D%0Aprint%28bajo_minimos%28inventario%2C%20150%29%29&mode=edit&py=3)

### Listas dentro de diccionarios

Los valores pueden ser de cualquier tipo, también listas. Así sabemos quién trabaja en cada módulo de la base:

In [ ]:
modulos = {
    "habitat": ["Ada", "Mae"],
    "laboratorio": ["Yuri"],
    "rover": ["Valentina"],
}
modulos["laboratorio"].append("Mae")    # Mae hace turno doble
modulos

[▶ Ver este ejemplo paso a paso en PythonTutor](https://pythontutor.com/render.html#code=modulos%20%3D%20%7B%0A%20%20%20%20%22habitat%22%3A%20%5B%22Ada%22%2C%20%22Mae%22%5D%2C%0A%20%20%20%20%22laboratorio%22%3A%20%5B%22Yuri%22%5D%2C%0A%7D%0Amodulos%5B%22laboratorio%22%5D.append%28%22Mae%22%29%0Aprint%28modulos%29&mode=edit&py=3)

Fijaos en que `modulos["laboratorio"]` **es** la lista, así que `append` la modifica dentro del diccionario. Al revés no funciona: una lista no puede ser **clave**. Queremos apuntar qué pareja tiene la guardia de noche:

In [ ]:
pareja = ["Ada", "Yuri"]
guardias = {}
guardias[pareja] = "noche"

`TypeError: unhashable type: 'list'`. Las claves tienen que ser **hashables**: Python calcula con la función `hash` un número entero a partir de la clave y lo usa para saber dónde guardarla. Si la clave pudiera cambiar (y las listas cambian), ese número dejaría de valer y no volveríamos a encontrarla. Por eso los strings y los enteros sirven como clave y las listas no.

In [ ]:
hash("agua"), hash(42)

### Acumular a partir de un diccionario

Otro filtro con el diccionario de palabras: los **palíndromos**, palabras que se leen igual al revés, como *noon*. Primero los contamos; ¿veis el patrón de siempre?

In [ ]:
def es_palindromo(palabra):
    return invertir(palabra) == palabra

cuenta = 0
for palabra in diccionario_palabras:
    if es_palindromo(palabra):
        cuenta += 1
cuenta

Si en lugar de un contador usamos una lista vacía y `append`, obtenemos los palíndromos. La lista hace de **acumulador**:

In [ ]:
palindromos = []
for palabra in diccionario_palabras:
    if es_palindromo(palabra):
        palindromos.append(palabra)
palindromos[:10]

In [ ]:
largos = []
for palabra in palindromos:
    if len(palabra) >= 7:
        largos.append(palabra)
largos

### Memos: que el ordenador de a bordo no repita cálculos

¿Os acordáis de `fibonacci` del capítulo 6? Cuanto mayor es `n`, más tarda, y mucho más deprisa de lo que crece `n`.

In [ ]:
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

%time fibonacci(30)

El problema es que repite trabajo: `fibonacci(30)` calcula `fibonacci(28)` dos veces, `fibonacci(27)` tres veces, y así hacia abajo. La solución es apuntar cada resultado la primera vez que lo calculamos en un diccionario, que hace de **memo**:

In [ ]:
conocidos = {0: 0, 1: 1}

def fibonacci_memo(n):
    if n in conocidos:
        return conocidos[n]          # ya lo sabíamos: no se calcula
    resultado = fibonacci_memo(n - 1) + fibonacci_memo(n - 2)
    conocidos[n] = resultado         # lo apuntamos para la próxima vez
    return resultado

%time fibonacci_memo(30)

En PythonTutor se ve cómo `conocidos` va creciendo y cómo, a partir de cierto punto, las llamadas ya encuentran la respuesta apuntada:

[▶ Ver este ejemplo paso a paso en PythonTutor](https://pythontutor.com/render.html#code=conocidos%20%3D%20%7B0%3A%200%2C%201%3A%201%7D%0A%0Adef%20fibonacci_memo%28n%29%3A%0A%20%20%20%20if%20n%20in%20conocidos%3A%0A%20%20%20%20%20%20%20%20return%20conocidos%5Bn%5D%0A%20%20%20%20resultado%20%3D%20fibonacci_memo%28n%20-%201%29%20%2B%20fibonacci_memo%28n%20-%202%29%0A%20%20%20%20conocidos%5Bn%5D%20%3D%20resultado%0A%20%20%20%20return%20resultado%0A%0Aprint%28fibonacci_memo%285%29%29%0Aprint%28conocidos%29&mode=edit&py=3)

Con el memo, hasta `fibonacci_memo(90)` sale al instante. Con la versión sin memo serían billones de llamadas: tardaríamos miles de años.

In [ ]:
fibonacci_memo(90)

### Depuración: cuando los datos son grandes

Con 100 000 palabras ya no podemos revisar la salida a ojo. Algunas ideas del capítulo:

1. **Reducir la entrada**: probad primero con unas pocas muestras, no con todo el archivo.
2. **Comprobar resúmenes y tipos**: en lugar de imprimirlo todo, mirad `len`, la suma o el `type` del resultado.
3. **Escribir autocomprobaciones**: código que avisa si algo no cuadra.
4. **Formatear la salida**: `pprint` muestra los diccionarios ordenados, uno por línea.

Un error típico al contar: olvidarse de inicializar el contador. ¿Qué pasa la primera vez que se ejecuta `contador[x] += 1`?

In [ ]:
def contar_mal(secuencia):
    contador = {}
    for x in secuencia:
        contador[x] += 1        # ¿y si x todavía no está?
    return contador

contar_mal(muestras)

`KeyError: 'basalto'`. `+= 1` primero tiene que **leer** el valor actual, y la clave aún no existe. Por eso `contar` comprueba con `in` antes de sumar. Una autocomprobación sencilla para `contar`: la suma de todos los recuentos tiene que coincidir con el número de muestras.

In [ ]:
recuento = contar(muestras)
sum(recuento.values()) == len(muestras)

In [ ]:
from pprint import pprint
pprint(contar("olympus mons"))

### Ejercicios de la parte 10

Primero, `tiene_repetidos`: queremos saber si alguien aparece dos veces en una lista de turnos, o si una palabra repite alguna letra. Pista: `contar` ya hace casi todo el trabajo.

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

def tiene_repetidos(t):
    """True si algún elemento de la secuencia t aparece más de una vez.

    >>> tiene_repetidos(['Ada', 'Mae', 'Ada'])
    True
    >>> tiene_repetidos('marte')
    False
    >>> tiene_repetidos('banana')
    True
    >>> tiene_repetidos([1, 2, 3])
    False
    """
    return None    # sustituye esta línea por tu solución

run_doctests(tiene_repetidos)

Mientras la función devuelva `None`, los doctests fallan y lo cuentan. Cuando estén todos en verde, `run_doctests` no imprime nada.

In [ ]:
# Ahora vosotros:
# 1. Con tiene_repetidos y la lista `palabras`, buscad las palabras de más de 12 letras
#    sin ninguna letra repetida. ¿Hay alguna más larga que "unpredictably"?
# 2. contar_con_get(secuencia): la misma función que contar, pero sin if.
#    Pista: inventario.get("cafe", 0) devuelve 0 si la clave no existe.
# 3. repetidos(contador): devuelve una lista con las claves cuyo recuento es mayor que 1.
#    repetidos(contar("banana")) -> ['a', 'n']
#    repetidos(contar(muestras)) -> ['basalto', 'olivino']
# 4. sumar_contadores(c1, c2): el rover sale dos días. Devuelve un diccionario NUEVO
#    con el total de cada tipo de roca.
#    sumar_contadores({"basalto": 2, "olivino": 1}, {"basalto": 1, "yeso": 3})
#    -> {'basalto': 3, 'olivino': 1, 'yeso': 3}
# 5. es_entrelazada(palabra): True si al tomar letras alternas salen dos palabras de la lista.
#    "schooled" -> "shoe" (palabra[0::2]) y "cold" (palabra[1::2]).
#    Usad diccionario_palabras para comprobarlo rápido.

## Para practicar en Codewars

Cuatro kata que salen con lo de hoy, de menos a más difícil. Algunas están pensadas para JavaScript y llaman *hash* u *object* a los diccionarios; en Python son `dict`:

- [**Welcome!**](https://www.codewars.com/kata/577ff15ad648a14b780000e7) (8 kyu): saludar en el idioma de cada astronauta. Un diccionario de idiomas a saludos y un `in` antes de buscar, para que no salte el `KeyError`.
- [**Isograms**](https://www.codewars.com/kata/54ba84be607a92aa900000f1) (7 kyu): una palabra sin letras repetidas. Es `tiene_repetidos` al revés (ojo con mayúsculas y minúsculas).
- [**Count characters in your string**](https://www.codewars.com/kata/52efefcbcdf57161d4000091) (6 kyu): contar cuántas veces aparece cada letra. Es `contar` tal cual.
- [**Find the odd int**](https://www.codewars.com/kata/54da5a58ea159efa38000836) (6 kyu): el número que aparece un número impar de veces. `contar` y luego recorrer el diccionario.

Reto extra: [**Memoized Fibonacci**](https://www.codewars.com/kata/529adbf7533b761c560004e5) (5 kyu). Si habéis entendido `fibonacci_memo`, ya la tenéis.

## Cierre

1. Un diccionario mapea **claves** a **valores**: se crea con `{}` o `dict()`, se rellena con `d[clave] = valor` y se consulta con `d[clave]`. Si la clave no existe, `KeyError`.
2. `in` mira las claves; para los valores, `d.values()`. Y es muy rápido, porque el diccionario es una tabla hash.
3. Un `for` sobre un diccionario recorre las claves. Contar cosas con un diccionario es el patrón *inicializar, recorrer, actualizar* de siempre.
4. Los diccionarios son mutables: `b = a` crea un alias y `dict(a)` una copia. Los valores pueden ser listas, pero las claves tienen que ser **hashables** (strings y enteros sí, listas no).
5. Un **memo** guarda resultados ya calculados para no repetirlos: `fibonacci_memo(90)` sale al instante.

Próxima parada: capítulo 11, tuplas. Veremos `items()`, que recorre claves y valores a la vez, y por qué una tupla sí puede ser clave de un diccionario.